In [1]:
## TO RUN ON THE CLOUD 

## preprocessed the data before starting the slide
## import embeddings <s
import os

# Define the URI to point to your manual process
os.environ["FIFTYONE_DATABASE_URI"] = "mongodb://localhost:44123"

import fiftyone as fo

# Verify connection
print(fo.core.odm.database.get_db_conn()) 


You are running the oldest supported major version of MongoDB. Please refer to https://deprecation.voxel51.com for deprecation notices. You can suppress this exception by setting your `database_validation` config parameter to `False`. See https://docs.voxel51.com/user_guide/config.html#configuring-a-mongodb-connection for more information
Database(MongoClient(host=['localhost:44123'], document_class=dict, tz_aware=False, connect=True, appname='fiftyone'), 'fiftyone')


In [2]:
import numpy as np 

In [3]:
import sys
%load_ext autoreload
%autoreload 2

In [2]:
fo.list_datasets()

['FLPLAN',
 'flplan200',
 'flplanpatches',
 'nc765',
 'nc765_T1024ov224',
 'tiles_merged',
 'umflplan_T1024ov224',
 'wp125',
 'wp125_T1024ov224']

In [4]:
dataset = fo.load_dataset('tiles_merged')


In [5]:
# load embeddings 
from sklearn.preprocessing import normalize

data_embedding_tile        = np.load("/share/home/e2406743/code/Dugongs_IRISA-MARBEC-LIRMM/chapter3/embeddings_dinov3/merged_1024o224_fullimage_embeddings.npz")

sample_ids = data_embedding_tile['sample_ids']

embedding_retrieved = data_embedding_tile['embeddings']

In [6]:
label_lookup = {
        s.id: s.get_field("type_label")
        for s in dataset.select_fields("type_label")
    }

In [7]:
dataset.count_values("dataset_sr")

{'UM_flplan': 2755, 'NC_flplan': 5728, 'WPmix': 2640}

In [7]:
from fiftyone import ViewField
nc_ids = dataset.match(
    ViewField("dataset_sr").contains_str("NC_flplan")
).values('id')
um_test_ids  = dataset.match(
    ViewField("dataset_sr").contains_str("UM_flplan")
).values('id') 

## look up the argidx of the given ids
mask_train = np.isin(sample_ids, nc_ids)
mask_test = np.isin(sample_ids, um_test_ids)

X_train = embedding_retrieved[mask_train]
X_test = embedding_retrieved[mask_test]

def _resolve_labels(ids):
        labels = []
        missing_label = []
        for sid in ids:
            lab = label_lookup.get(sid)
            if lab == "positive":
                labels.append(1)
            elif lab == "negative":
                labels.append(0)
            else:
                labels.append(-1)   # sentinel for "not found / unexpected value"
                missing_label.append(sid)
        return np.array(labels), missing_label



y_train, missing_train = _resolve_labels(nc_ids)
y_test,  missing_test  = _resolve_labels(um_test_ids)

#normalize
X_train = normalize(X_train, norm="l2")
X_test  = normalize(X_test,  norm="l2")

In [16]:
sample_ids

array(['6a33c2d08be790ca1e89a2ac', '6a33c2d08be790ca1e89a2ad',
       '6a33c2d08be790ca1e89a2ae', ..., '6a33c5828be790ca1e89c57f',
       '6a33c5828be790ca1e89c580', '6a33c5828be790ca1e89c581'],
      shape=(11123,), dtype='<U24')

In [8]:
X_train.shape

(5728, 1024)

In [9]:
X_test.shape

(2755, 1024)

In [10]:
def _resolve_labels(ids):
        labels = []
        missing_label = []
        for sid in ids:
            lab = label_lookup.get(sid)
            if lab == "positive":
                labels.append(1)
            elif lab == "negative":
                labels.append(0)
            else:
                labels.append(-1)   # sentinel for "not found / unexpected value"
                missing_label.append(sid)
        return np.array(labels), missing_label



y_train, missing_train = _resolve_labels(nc_ids)
y_test,  missing_test  = _resolve_labels(um_test_ids)

In [22]:
y_train.shape

(5728,)

In [23]:
y_test.shape

(2755,)

In [11]:
#normalize
X_train = normalize(X_train, norm="l2")
X_test  = normalize(X_test,  norm="l2")

In [12]:
from src.classification import fit_logistic_regression, evaluate_predictions, fit_random_forest

In [26]:
result  = fit_logistic_regression(
    X_train, y_train, X_test, y_test, test_ids=um_test_ids
)


=== Logistic Regression ===
  Accuracy : 0.9655
  Precision: 0.6106
  Recall   : 0.9007
  F1       : 0.7278
  ROC-AUC  : 0.9836
  Confusion matrix [[TN,FP],[FN,TP]]:
[[2533   81]
 [  14  127]]
              precision    recall  f1-score   support

    negative       0.99      0.97      0.98      2614
    positive       0.61      0.90      0.73       141

    accuracy                           0.97      2755
   macro avg       0.80      0.93      0.85      2755
weighted avg       0.97      0.97      0.97      2755



In [13]:
from src.classification import fit_logistic_regression, fit_random_forest

#result_lr = fit_logistic_regression(X_train, y_train, X_test, y_test, test_ids=um_test_ids)
result_rf = fit_random_forest(X_train, y_train, X_test, y_test, test_ids=um_test_ids)

# Visualize LR failures directly in the App
#session.view = dataset.select(result_lr["failed_ids"])

# Combine both models' results into one DataFrame for plotting
#import pandas as pd
#all_results = pd.concat([result_lr["results_df"], result_rf["results_df"]], ignore_index=True)


=== Random Forest ===
  Accuracy : 0.9858
  Precision: 0.8643
  Recall   : 0.8582
  F1       : 0.8612
  ROC-AUC  : 0.9862
  Confusion matrix [[TN,FP],[FN,TP]]:
[[2595   19]
 [  20  121]]
              precision    recall  f1-score   support

    negative       0.99      0.99      0.99      2614
    positive       0.86      0.86      0.86       141

    accuracy                           0.99      2755
   macro avg       0.93      0.93      0.93      2755
weighted avg       0.99      0.99      0.99      2755

  Failed predictions: 39 / 2755
